In [55]:
import pandas as pd
import numpy as np
import re

In [56]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_7\messy_inventory.csv")

In [57]:
df.shape

(39, 15)

In [58]:
df.dtypes

SKU                    str
ProductName            str
Category               str
Weight                 str
ListPrice              str
DiscountPercent        str
Price                  str
StockQty             int64
WarehouseCode        int64
Supplier               str
RestockDate            str
IsActive               str
Rating             float64
OriginCountry          str
Notes                  str
dtype: object

In [59]:
df.head(5)

,SKU,ProductName,Category,Weight,ListPrice,DiscountPercent,Price,StockQty,WarehouseCode,Supplier,RestockDate,IsActive,Rating,OriginCountry,Notes
0,SKU-1001-RED-L,Classic Cotton T-Shirt,Apparel,180g,$19.99,10%,$17.99,120,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
1,SKU-1001-BLU-M,Classic Coton T-Shirt,apparel,0.18kg,$19.99,10%,$17.99,85,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
2,SKU-1002-BLK-S,classic cotton tshirt,Apparel,180g,$19.99,0%,$19.99,60,234,Fabrico,2023-06,Yes,4.5,USA,NaN
3,SKU-1003-GRN-XL,Slim Fit Denim Jeans,Apparel,650g,$54.99,20%,$43.99,45,234,Denimworks,2023-07-01,Yes,4.2,USA,NaN
4,SKU-1004-BLK-M,Slim-Fit Denim Jeans,apparel,0.65kg,$54.99,20%,$43.99,30,234,Denimworks,2023-07,yes,4.2,USA,Name variant


## `SKU`: multiple capture groups in one regex

**Decision:** `SKU-1234-RED-L` has three distinct pieces of information glued together. `.str.extract()` with three parenthesized groups pulls all three out in a single pass, `(\d+)` for the numeric ID, `([A-Z]+)` for the color, `([A-Z]+)` for the size.

In [60]:
sku_split = df["SKU"].str.extract(r"SKU-(\d+)-([A-Z]+)-([A-Z]+)")
df["ProductID"] = sku_split[0]
df["Color"] = sku_split[1]
df["Size"] = sku_split[2]

df[["SKU", "ProductID", "Color", "Size"]]

,SKU,ProductID,Color,Size
0,SKU-1001-RED-L,1001,RED,L
1,SKU-1001-BLU-M,1001,BLU,M
2,SKU-1002-BLK-S,1002,BLK,S
3,SKU-1003-GRN-XL,1003,GRN,XL
4,SKU-1004-BLK-M,1004,BLK,M
5,SKU-1005-WHT-L,1005,WHT,L
6,SKU-1006-BLK-L,1006,BLK,L
7,SKU-1007-SLV-M,1007,SLV,M
8,SKU-1008-BLU-M,1008,BLU,M
9,SKU-1009-BLK-S,1009,BLK,S


## Column Decomposition & Reordering

**Objective:**  
Replace the monolithic `SKU` column with more granular product attribute fields (`ProductID`, `Color`, `Size`) positioned directly where the `SKU` column originally sat.

**Fix Applied:**  
1. **Extracted Attributes:** Parsed individual product attributes (`ProductID`, `Color`, `Size`) from the raw `SKU` values.
2. **Repositioned Columns:** Placed the three new columns at the precise index location of the original `SKU` column.
3. **Removed Redundant Field:** Dropped the `SKU` column to complete the structural update.

In [61]:
target = df.columns.get_loc("SKU")

df.insert(target , "ProductID", df.pop("ProductID"))
df.insert(target + 1, "Color", df.pop("Color"))
df.insert(target + 2, "Size", df.pop("Size"))
df.drop(columns = "SKU", inplace = True)

In [62]:
df["ProductName"] = df["ProductName"].astype(str).str.strip().str.lower()
df["ProductName"]

0           classic cotton t-shirt
1            classic coton t-shirt
2            classic cotton tshirt
3             slim fit denim jeans
4             slim-fit denim jeans
5       wireless bluetooth earbuds
6       wireless bluetooth earbuds
7     stainless steel water bottle
8     stainless steel water bottle
9                 yoga mat premium
10                yoga mat premium
11                  leather wallet
12                  leather wallet
13               running shoes pro
14               running shoes pro
15              ceramic coffee mug
16              ceramic coffee mug
17          bluetooth speaker mini
18          bluetooth speaker mini
19                kids rain jacket
20                kids rain jacket
21         adjustable dumbbell set
22         adjustable dumbbell set
23              desk organizer set
24              desk organizer set
25             insulated lunch bag
26             insulated lunch bag
27        mens cotton socks 3-pack
28       men's cotto

In [63]:
df["Category"] = df["Category"].astype(str).str.strip().str.title()
df["Category"]

0         Apparel
1         Apparel
2         Apparel
3         Apparel
4         Apparel
5     Electronics
6     Electronics
7            Home
8            Home
9         Fitness
10        Fitness
11    Accessories
12    Accessories
13       Footwear
14       Footwear
15           Home
16           Home
17    Electronics
18    Electronics
19        Apparel
20        Apparel
21        Fitness
22        Fitness
23           Home
24           Home
25           Home
26           Home
27        Apparel
28        Apparel
29    Electronics
30    Electronics
31        Outdoor
32        Outdoor
33           Home
34           Home
35        Apparel
36        Apparel
37        Outdoor
38        Outdoor
Name: Category, dtype: str

## Weight Normalization to Kg.

**Issue Identified:**  
The weight data contains mixed measurement units (`kg` and `gm`/`g`), preventing direct quantitative analysis and numerical aggregation.

**Fix Applied:**  
1. **Parsed Numerical Values & Units:** Extracted numeric values alongside their respective unit indicators.
2. **Unit Conversion:** Converted all gram values (`g/gm`) to kilogram (`kg`) by dividing by $1000$.
3. **Type Conversion:** Formatted the final result into a uniform numeric column represented purely in kg (`kg`).

In [64]:
def clean_weight(val):
    val = str(val).strip().lower()
    if val in ("nan", "n", ""):
        return np.nan
    num_match = re.search(r"([\d.]+)", val)
    if not num_match:
        return np.nan
    num = float(num_match.group(0))
    if "kg" in val:
        return f"{num} kg"
    elif "g" in val:
        return f"{round(num/1000,2)} kg"
    return val

df["Weight"] = df["Weight"].apply(clean_weight)
df["Weight"] = df["Weight"].astype(str).str.strip()
df["Weight"]

0      0.18 kg
1      0.18 kg
2      0.18 kg
3      0.65 kg
4      0.65 kg
5      0.06 kg
6     0.055 kg
7       0.4 kg
8       0.4 kg
9       1.2 kg
10      1.2 kg
11     0.09 kg
12     0.09 kg
13     0.85 kg
14     0.85 kg
15     0.32 kg
16     0.32 kg
17     0.21 kg
18     0.21 kg
19      0.3 kg
20      0.3 kg
21     15.0 kg
22     15.0 kg
23     0.55 kg
24     0.55 kg
25     0.25 kg
26     0.25 kg
27     0.12 kg
28     0.12 kg
29      0.1 kg
30    0.095 kg
31      3.2 kg
32      3.2 kg
33      0.9 kg
34      0.9 kg
35     0.08 kg
36     0.08 kg
37      1.1 kg
38      1.1 kg
Name: Weight, dtype: str

## Price & Discount Character Removal

**Issue Identified:**  
The `ListPrice` and `DiscountPercent` columns contain currency symbols (`$`) and percentage signs (`%`), storing numeric data as strings (`object`) and preventing financial calculations.

**Fix Applied:**  
1. **Stripped Symbols:** Removed `$` from `ListPrice` and `%` from `DiscountPercent` using regex/string stripping.
2. **Type Conversion:** Converted `ListPrice` to numeric float (`float64`) and `DiscountPercent` to float (`float64`).
3. **Prepared for Calculations:** Standardized values to allow direct numerical operations (e.g., calculating net price).

In [65]:
df["ListPrice"] = df["ListPrice"].astype(str).str.strip().str.replace(r'[^\d.]', "", regex=True)
df["ListPrice"].head(5)

0    19.99
1    19.99
2    19.99
3    54.99
4    54.99
Name: ListPrice, dtype: str

In [66]:
df["DiscountPercent"] = df["DiscountPercent"].astype(str).str.strip().str.replace(r"[^\d.]", "", regex=True)
df["DiscountPercent"]

0     10
1     10
2      0
3     20
4     20
5     15
6     15
7      0
8     10
9     25
10    25
11     0
12     0
13    30
14    30
15     0
16     0
17    20
18    20
19     0
20    10
21    10
22    10
23     0
24     5
25     0
26    15
27     0
28     0
29     0
30    10
31    20
32    20
33     0
34     5
35     0
36     0
37    15
38    15
Name: DiscountPercent, dtype: str

## 𝄈 Data Validation: Discounted Price Verification

**Objective:**  
Cross-check the final listed sale price against the calculated net price based on `ListPrice` and `DiscountPercent` to detect pricing errors, system rounding discrepancies, or mismatches.

**Validation Formula:**  
$$\text{Expected Price} = \text{ListPrice} \times \left(1 - \frac{\text{DiscountPercent}}{100}\right)$$

**Fix & Verification Applied:**  
1. **Computed Expected Price:** Calculated the exact discounted amount for every record (rounded to 2 decimal places).
2. **Flagged Mismatches:** Compared `Expected Price` against `Price` using a small tolerance threshold (e.g., $0.01$) to account for potential floating-point rounding differences.
3. **Identified Pricing Errors:** Isolated any rows where the listed `Price` deviates from the calculated discounted price.

In [67]:
df.dtypes

ProductID              str
Color                  str
Size                   str
ProductName            str
Category               str
Weight                 str
ListPrice              str
DiscountPercent        str
Price                  str
StockQty             int64
WarehouseCode        int64
Supplier               str
RestockDate            str
IsActive               str
Rating             float64
OriginCountry          str
Notes                  str
dtype: object

In [68]:
# The `ListPrice` & `DiscountPercent` dtypes are str, have to convert them to numeric first.

df["ListPrice"] = pd.to_numeric(df["ListPrice"], errors='coerce')
df["DiscountPercent"] = pd.to_numeric(df["DiscountPercent"], errors='coerce')


In [69]:
# Before cross checking, `$` needs to remove from the price and then convert the column to numeric.

df["Price"] = df["Price"].astype(str).str.strip().str.replace(r'[^\d.]', "", regex=True)
df["Price"] = pd.to_numeric(df["Price"], errors='coerce')
df["Price"].head(5)

0    17.99
1    17.99
2    19.99
3    43.99
4    43.99
Name: Price, dtype: float64

In [70]:
df["PriceCheck"] = (df["ListPrice"] * (1 - (df["DiscountPercent"] / 100))).round(2)
df["PriceCheck"]

0      17.99
1      17.99
2      19.99
3      43.99
4      43.99
5      67.99
6      67.99
7      24.99
8      22.49
9      26.24
10     26.24
11     45.00
12     45.00
13     62.99
14     62.99
15     12.99
16     12.99
17     31.99
18     31.99
19     29.99
20     26.99
21    134.99
22    134.99
23     22.50
24     21.38
25     18.99
26     16.14
27     14.99
28     14.99
29     25.99
30     23.39
31     95.99
32     95.99
33     32.99
34     31.34
35     16.99
36     16.99
37     55.24
38     55.24
Name: PriceCheck, dtype: float64

In [71]:
mismatch_price = (df["Price"] != df["PriceCheck"])
df.loc[mismatch_price, ["ProductID", "Category", "ListPrice", "PriceCheck", "Price"]]

,ProductID,Category,ListPrice,PriceCheck,Price
8,1008,Home,24.99,22.49,21.99
10,1010,Fitness,34.99,26.24,29.24
18,1018,Electronics,39.99,31.99,27.99


## Resolving Price Mismatches

**Issue Identified:**  
Cross-validation between the original `Price` column and calculated `PriceCheck` values revealed discrepancies in three specific products.

**Fix Applied:**  
1. **Replaced Mismatched Values:** Updated the original price records by filling missing or incorrect values in `Price` using the verified `PriceCheck` calculations.
2. **Maintained Data Consistency:** Standardized the product pricing column (`Price`) across all rows to reflect accurate discounted rates.

In [72]:
df["Price"] = np.where(df["Price"] != df["PriceCheck"], df["PriceCheck"], df["Price"])

In [73]:
# Check again if the mismatches were updated.

mismatch_price = (df["Price"] != df["PriceCheck"])
df.loc[mismatch_price, ["ProductID", "Category", "ListPrice", "PriceCheck", "Price"]]

,ProductID,Category,ListPrice,PriceCheck,Price


> Removing the `PriceCheck` column.

In [74]:
df = df.drop(columns="PriceCheck")

In [75]:
df.head(5)

,ProductID,Color,Size,ProductName,Category,Weight,ListPrice,DiscountPercent,Price,StockQty,WarehouseCode,Supplier,RestockDate,IsActive,Rating,OriginCountry,Notes
0,1001,RED,L,classic cotton t-shirt,Apparel,0.18 kg,19.99,10,17.99,120,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
1,1001,BLU,M,classic coton t-shirt,Apparel,0.18 kg,19.99,10,17.99,85,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
2,1002,BLK,S,classic cotton tshirt,Apparel,0.18 kg,19.99,0,19.99,60,234,Fabrico,2023-06,Yes,4.5,USA,NaN
3,1003,GRN,XL,slim fit denim jeans,Apparel,0.65 kg,54.99,20,43.99,45,234,Denimworks,2023-07-01,Yes,4.2,USA,NaN
4,1004,BLK,M,slim-fit denim jeans,Apparel,0.65 kg,54.99,20,43.99,30,234,Denimworks,2023-07,yes,4.2,USA,Name variant


## Restoring Currency & Percent Symbols

**Objective:**  
Now that pricing has been cross-checked and mismatches corrected, re-apply the display symbols (`$` on `ListPrice`/`Price`, `%` on `DiscountPercent`).

**Fix Applied:**  
1. **Formatted With Symbols:** Used `.map()` with an f-string style template to attach the symbol and lock prices to 2 decimal places.
2. **Note on Dtype:** These columns become strings (`str`) again, so run this cell **once**, and only after all numeric work is finished.

In [76]:
def dollar_symbol_add(val):
    if pd.isna(val) or val == "nan":
        return np.nan
    val = str(val)
    return "$" + val

def percentage_symbol_add(val):
    if pd.isna(val) or val == "nan":
        return np.nan
    val = str(val)
    return val + "%"

df["ListPrice"] = df["ListPrice"].apply(dollar_symbol_add)
df["Price"] = df["Price"].apply(dollar_symbol_add)
df["DiscountPercent"] = df["DiscountPercent"].apply(percentage_symbol_add)

In [77]:
df.head(5)

,ProductID,Color,Size,ProductName,Category,Weight,ListPrice,DiscountPercent,Price,StockQty,WarehouseCode,Supplier,RestockDate,IsActive,Rating,OriginCountry,Notes
0,1001,RED,L,classic cotton t-shirt,Apparel,0.18 kg,$19.99,10%,$17.99,120,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
1,1001,BLU,M,classic coton t-shirt,Apparel,0.18 kg,$19.99,10%,$17.99,85,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
2,1002,BLK,S,classic cotton tshirt,Apparel,0.18 kg,$19.99,0%,$19.99,60,234,Fabrico,2023-06,Yes,4.5,USA,NaN
3,1003,GRN,XL,slim fit denim jeans,Apparel,0.65 kg,$54.99,20%,$43.99,45,234,Denimworks,2023-07-01,Yes,4.2,USA,NaN
4,1004,BLK,M,slim-fit denim jeans,Apparel,0.65 kg,$54.99,20%,$43.99,30,234,Denimworks,2023-07,yes,4.2,USA,Name variant


## Invalid Inventory Value Treatment

**Issue Identified:**  
The `StockQty` column contains negative inventory values (e.g., `-15`), which represent logically invalid physical stock counts.

**Fix Applied:**  
1. **Identified Non-Positive Values:** Flagged negative stock numbers using conditional filtering (`StockQty < 0`).
2. **Coerced to Missing Data:** Replaced invalid negative values with `np.nan`.
3. **Status:** Update the Active status to `No`.

In [78]:
df.loc[df["StockQty"] < 0, ["ProductID", "ProductName", "StockQty", "Notes"]]

,ProductID,ProductName,StockQty,Notes
16,1016,ceramic coffee mug,-15,Negative stock?


In [79]:
df.loc[df["StockQty"] < 0, "StockQty"] = np.nan

In [80]:
df.loc[df["StockQty"] < 0, ["ProductID", "ProductName", "StockQty", "Notes"]]

,ProductID,ProductName,StockQty,Notes


In [81]:
df.loc[df["StockQty"].isna(), "IsActive"] = "No"

In [82]:
df["IsActive"]

0     Yes
1     Yes
2     Yes
3     Yes
4     yes
5     Yes
6     Yes
7     Yes
8     Yes
9     Yes
10    Yes
11    Yes
12    Yes
13    Yes
14    Yes
15    Yes
16     No
17    Yes
18    Yes
19    Yes
20    Yes
21    Yes
22    Yes
23    Yes
24    Yes
25    Yes
26    Yes
27    Yes
28    Yes
29     No
30     No
31    Yes
32    Yes
33    Yes
34    Yes
35    Yes
36    Yes
37    Yes
38    Yes
Name: IsActive, dtype: str

## RestockDate Normalization & Datetime Parsing

**Issue Identified:**  
The `RestockDate` column contains mixed resolution string formats (full dates `YYYY-MM-DD`, year-month `YYYY-MM`, and year-only `YYYY`), preventing chronological sorting and time-series operations.

**Fix Applied:**  
1. **Coerced to Datetime:** Used `pd.to_datetime()` with `errors='coerce'` to parse string dates into standardized datetime objects.
2. **Standardized Incomplete Dates:** Automatically defaults incomplete date structures (e.g., `YYYY-MM` or `YYYY`) to the first day of that period (e.g., `2023-06` $\rightarrow$ `2023-06-01`).

In [83]:
df["RestockDate"] = pd.to_datetime(df["RestockDate"], format='mixed', errors='coerce').dt.strftime("%Y-%m-%d")
df["RestockDate"].head(10)

0    2023-06-15
1    2023-06-15
2    2023-06-01
3    2023-07-01
4    2023-07-01
5    2023-05-20
6    2023-05-20
7    2023-08-10
8    2023-08-01
9    2023-04-15
Name: RestockDate, dtype: str

## issing Value Normalization in Notes Column

**Issue Identified:**  
The `Notes` column contains string representations of missing values (e.g., `"nan"`, `"na"`, and blank strings `""`) stored as text instead of proper null indicators.

**Fix Applied:**  
1. **Normalized Null Representations:** Used `.replace()` to convert pseudo-null text variants (`"nan"`, `"na"`, `""`) directly to `np.nan`.

In [84]:
df["Notes"] = df["Notes"].astype(str).str.strip().replace(["nan","na",""],np.nan)
df["Notes"]

0                        NaN
1                        NaN
2                        NaN
3                        NaN
4               Name variant
5                        NaN
6           Exact duplicate?
7                        NaN
8                        NaN
9                        NaN
10           Price mismatch?
11            Year-only date
12    Duplicate of SKU-1011?
13                       NaN
14          Exact duplicate?
15                       NaN
16           Negative stock?
17                       NaN
18           Price mismatch?
19                       NaN
20                       NaN
21                       NaN
22          Exact duplicate?
23                       NaN
24                       NaN
25                       NaN
26                       NaN
27                       NaN
28              Name variant
29                       NaN
30                       NaN
31                       NaN
32          Exact duplicate?
33                       NaN
34            

> Origin Country count for customers

In [85]:
df["OriginCountry"].value_counts()

OriginCountry
USA        29
China       6
Italy       2
Vietnam     2
Name: count, dtype: int64

## Duplicate Identification & Removal

**Issue Identified:**  
Identical duplicate records across the dataset can skew analytical metrics, double-count transactions, and inflate summary statistics.

**Fix Applied:**  
1. **Identified Duplicates:** Scanned the DataFrame using `df.duplicated()` to check for redundant rows.
2. **Removed Redundancies:** Dropped duplicate rows using `df.drop_duplicates()`, retaining the first occurrence.
3. **Reset Index:** Reset the DataFrame index (`reset_index(drop=True)`) to maintain a clean, continuous sequence post-removal.

In [86]:
dup_cols = ["ProductID", "ProductName", "Category", "Weight", "ListPrice", "Price", "RestockDate"]

df[df.duplicated(subset=dup_cols, keep=False)]

,ProductID,Color,Size,ProductName,Category,Weight,ListPrice,DiscountPercent,Price,StockQty,WarehouseCode,Supplier,RestockDate,IsActive,Rating,OriginCountry,Notes


**No real duplicate but in the dataset there is comment "Exact duplicates?", are those real duplicate, let's check**

In [87]:
flagged = df[df["Notes"].astype(str).str.contains("Exact duplicate", na = False)]

flagged[["ProductID", "ProductName", "Color", "Size", "Notes"]]

,ProductID,ProductName,Color,Size,Notes
6,1006,wireless bluetooth earbuds,BLK,L,Exact duplicate?
14,1014,running shoes pro,RED,M,Exact duplicate?
22,1022,adjustable dumbbell set,GRY,L,Exact duplicate?
32,1032,camping tent 2-person,GRN,L,Exact duplicate?


> No they are not. Drop the note and replace them with np.nan

In [88]:
df.loc[df["Notes"] == "Exact duplicate?", "Notes"] = np.nan

In [89]:
df["Notes"]

0                        NaN
1                        NaN
2                        NaN
3                        NaN
4               Name variant
5                        NaN
6                        NaN
7                        NaN
8                        NaN
9                        NaN
10           Price mismatch?
11            Year-only date
12    Duplicate of SKU-1011?
13                       NaN
14                       NaN
15                       NaN
16           Negative stock?
17                       NaN
18           Price mismatch?
19                       NaN
20                       NaN
21                       NaN
22                       NaN
23                       NaN
24                       NaN
25                       NaN
26                       NaN
27                       NaN
28              Name variant
29                       NaN
30                       NaN
31                       NaN
32                       NaN
33                       NaN
34            

In [90]:
df.shape

(39, 17)

In [91]:
df.dtypes

ProductID              str
Color                  str
Size                   str
ProductName            str
Category               str
Weight                 str
ListPrice              str
DiscountPercent        str
Price                  str
StockQty           float64
WarehouseCode        int64
Supplier               str
RestockDate            str
IsActive               str
Rating             float64
OriginCountry          str
Notes                  str
dtype: object

In [92]:
df.to_csv("Clean_Inventory_Data.csv", index=False)